In [22]:
import sys
from datetime import datetime, timedelta
from pathlib import Path

import chardet
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import seaborn as sns
from folium.plugins import HeatMap, MarkerCluster

# adds parent file of the current directory
# to the paths in which Python looks for modules to import
# in the current Python process
sys.path.append(str(Path.cwd().parent))

from src.config import DATA_RAW_DIR, GEO_DATA_RAW_DIR, GEO_DATA_CLEAN_DIR
from utils.cleaning_utils import delete, normalize_columns_names, normalize_text_columns_cells, optimize_numeric_column, fill_rate
from utils.analysis_utils import plot_missing_bar, plot_numeric_histograms, plot_corr_heatmap, plot_qualitative


In [23]:
pd.set_option('display.max_columns', None)

In [24]:
# List all CSV files in DATA_RAW
ALL_DATA_FILES = list(iter(DATA_RAW_DIR.glob('*.csv')))

# List all recent CSV files in DATA_RAW, post 2006, without 2003 file
POST_2005_DATA_FILES = list(iter(DATA_RAW_DIR.glob('Incendies20*.csv')))
POST_2005_DATA_FILES = [f for f in POST_2005_DATA_FILES if "2003" and "2001" not in str(f)]

RAW_DATA_FILE = GEO_DATA_RAW_DIR / 'communes-france-2025.csv'

RAW_DATA_FILE

PosixPath('/home/coule/Documents/projets/incendies/data/data_geo_raw/communes-france-2025.csv')

In [25]:
# Check the encoding of the CSV files
with open(RAW_DATA_FILE, 'rb') as file:
    encodage = chardet.detect(file.read(10000))

print(encodage)

{'encoding': 'utf-8', 'confidence': 0.833516, 'language': 'fr', 'mime_type': 'text/plain'}


In [26]:
df = pd.read_csv(RAW_DATA_FILE, encoding=encodage["encoding"], dtype_backend='numpy_nullable')

/tmp/ipykernel_49007/650886494.py:1: DtypeWarning: Columns (0: code_insee, 1: dep_code, 2: canton_code, 3: epci_code, 4: code_insee_centre_zone_emploi, 5: code_unite_urbaine) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_DATA_FILE, encoding=encodage["encoding"], dtype_backend='numpy_nullable')


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34935 entries, 0 to 34934
Data columns (total 47 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   numero                             34935 non-null  Int64  
 1   code_insee                         34935 non-null  object 
 2   nom_standard                       34935 non-null  string 
 3   nom_sans_pronom                    34935 non-null  string 
 4   nom_a                              34935 non-null  string 
 5   nom_de                             34935 non-null  string 
 6   nom_sans_accent                    34935 non-null  string 
 7   nom_standard_majuscule             34935 non-null  string 
 8   typecom                            34935 non-null  string 
 9   typecom_texte                      34935 non-null  string 
 10  reg_code                           34935 non-null  Int64  
 11  reg_nom                            34935 non-null  string 
 12  d

In [28]:
df.columns

Index(['numero', 'code_insee', 'nom_standard', 'nom_sans_pronom', 'nom_a',
       'nom_de', 'nom_sans_accent', 'nom_standard_majuscule', 'typecom',
       'typecom_texte', 'reg_code', 'reg_nom', 'dep_code', 'dep_nom',
       'canton_code', 'canton_nom', 'epci_code', 'epci_nom', 'academie_code',
       'academie_nom', 'code_postal', 'codes_postaux', 'zone_emploi',
       'code_insee_centre_zone_emploi', 'code_unite_urbaine',
       'nom_unite_urbaine', 'taille_unite_urbaine',
       'type_commune_unite_urbaine', 'statut_commune_unite_urbaine',
       'population', 'superficie_hectare', 'superficie_km2', 'densite',
       'altitude_moyenne', 'altitude_minimale', 'altitude_maximale',
       'latitude_mairie', 'longitude_mairie', 'latitude_centre',
       'longitude_centre', 'grille_densite', 'grille_densite_texte',
       'niveau_equipements_services', 'niveau_equipements_services_texte',
       'gentile', 'url_wikipedia', 'url_villedereve'],
      dtype='str')

### Sélection/renommage/typage des variables
- altitude
'altitude_centre' à des NaN, on le supprime<br>
'altitude_mairie'  devient 'altitude'<br>

- code_postal<br>
    renommé 'localisation'

- region<br>
    'reg_code' devien: 'region'

- departement<br>
    'dep_code' devient 'departement'

In [29]:
# Dictionnaire de correspondance (Ancien nom -> Nouveau nom pour tes analyses)
colonnes_mapping = {
    'code_insee': 'code_insee',
    'code_postal': 'localisation',       # Correspond au code postal principal
    'nom_standard': 'nom_standard',
    'reg_code': 'region',                # Renommé pour l'analyse
    'dep_code': 'departement',           # Renommé pour l'analyse
    'population': 'population',
    'superficie_hectare': 'superficie_hectare',
    'densite': 'densite',
    'altitude_moyenne': 'altitude_moyenne',
    'altitude_minimale': 'altitude_minimale',
    'altitude_maximale': 'altitude_maximale',
    'latitude_mairie': 'latitude',
    'longitude_mairie': 'longitude'
}

# filtre le DataFrame d'origine pour ne garder que ces colonnes et les renommer
df_filtre = df[list(colonnes_mapping.keys())].rename(columns=colonnes_mapping)

# Définit les types cibles compatibles avec PostgreSQL
# Utiliser des majuscules (Int32, Float64, string) permet à Pandas de gérer les valeurs absentes (NaN) sans bloquer
types_cibles = {
    'code_insee': 'string',
    'localisation': 'Int32',
    'nom_standard': 'string',
    'region': 'Int8',
    'departement': 'string',  # En string pour garder les préfixes ou formats spéciaux si besoin
    'population': 'Int32',
    'superficie_hectare': 'Int32',
    'densite': 'Float64',
    'altitude_moyenne': 'Int16',
    'altitude_minimale': 'Int16',
    'altitude_maximale': 'Int16',
    'latitude': 'Float64',
    'longitude': 'Float64'
}

# applique le typage au DataFrame
df = df_filtre.astype(types_cibles)

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34935 entries, 0 to 34934
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code_insee          34935 non-null  string 
 1   localisation        34932 non-null  Int32  
 2   nom_standard        34935 non-null  string 
 3   region              34935 non-null  Int8   
 4   departement         34935 non-null  string 
 5   population          34935 non-null  Int32  
 6   superficie_hectare  34935 non-null  Int32  
 7   densite             34932 non-null  Float64
 8   altitude_moyenne    34935 non-null  Int16  
 9   altitude_minimale   34935 non-null  Int16  
 10  altitude_maximale   34935 non-null  Int16  
 11  latitude            34935 non-null  Float64
 12  longitude           34935 non-null  Float64
dtypes: Float64(3), Int16(3), Int32(3), Int8(1), string(3)
memory usage: 3.2 MB


In [31]:
df.head()

,code_insee,localisation,nom_standard,region,departement,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude,longitude
0,01001,1400,L'Abergement-Clémenciat,84,01,832,1565,53.0,242,206,272,46.151,4.921
1,01002,1640,L'Abergement-de-Varey,84,01,267,912,29.0,483,290,748,46.007,5.423
2,01004,1500,Ambérieu-en-Bugey,84,01,14854,2448,607.0,379,237,753,45.958,5.36
3,01005,1330,Ambérieux-en-Dombes,84,01,1897,1605,118.0,290,265,302,45.996,4.903
4,01006,1300,Ambléon,84,01,113,602,19.0,589,330,940,45.748,5.601


## Export

In [32]:
df.to_parquet(
    GEO_DATA_CLEAN_DIR / "communes.parquet",
    index=False
)